In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google import genai
from google.genai.errors import ServerError
from langchain_community.document_loaders import PyPDFLoader
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

C:\Users\Admin\AppData\Local\Temp\ipykernel_23676\2664611870.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader('attention.pdf')
docs = loader.load()
# print(docs)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
documents = text_splitter.split_documents(docs)

# Corrected variable name slice (documents instead of document)
print(documents[:1])

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszka

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Create FAISS vector database from your documents
db = FAISS.from_documents(
    documents=documents, 
    embedding=GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        # google_api_key=""
    )
)

print("FAISS vector database created successfully!")

In [ ]:
# !pip install -q langchain-community faiss-cpu langchain-google-genai

In [ ]:
query = "who are the authors of attention is all you need?"
import json
# retrived_result = db.similarity_search(query)
results = db.similarity_search_with_score(query, k=1)
# print(retrived_result[0].page_content)
# Format the results into a Python list of dictionaries
structured_data = []
for doc, score in results:
    structured_data.append({
        "content": doc.page_content,
        "metadata": doc.metadata,
        "score": float(score)
    })

# Convert to a clean JSON string and output it
json_output = json.dumps(structured_data, indent=4)
print(json_output)

In [4]:
import lancedb
from langchain_community.vectorstores import LanceDB
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Initialize local LanceDB connection directory
import tempfile
db_dir = tempfile.mkdtemp()
db = lancedb.connect(db_dir)

# 2. Create LanceDB vector database from your documents
dbw = LanceDB.from_documents(
    documents=documents, 
    embedding=GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        # google_api_key=""
    ),
    connection=db
)

print("LanceDB vector database created successfully!")

LanceDB vector database created successfully!


In [ ]:
!pip install -q lancedb

In [5]:
query = "who are the authors of attention is all you need?"
import json
# retrived_result = db.similarity_search(query)
results = dbw.similarity_search_with_score(query, k=1)
# print(retrived_result[0].page_content)
# Format the results into a Python list of dictionaries
structured_data = []
for doc, score in results:
    structured_data.append({
        "content": doc.page_content,
        "metadata": doc.metadata,
        "score": float(score)
    })

# Convert to a clean JSON string and output it
json_output = json.dumps(structured_data, indent=4)
print(json_output)

[
    {
        "content": "Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani\u2217\nGoogle Brain\navaswani@google.com\nNoam Shazeer\u2217\nGoogle Brain\nnoam@google.com\nNiki Parmar\u2217\nGoogle Research\nnikip@google.com\nJakob Uszkoreit\u2217\nGoogle Research\nusz@google.com\nLlion Jones\u2217\nGoogle Research\nllion@google.com\nAidan N. Gomez\u2217\u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\u0141ukasz Kaiser\u2217\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin\u2217\u2021\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, 